In [1]:
"""from datasets import load_dataset

# Loading only 1000 stories
dataset = load_dataset("roneneldan/TinyStories", split="train[:1000]")

# Saving everything as a text file
with open("data/tinystories_extract.txt", "w", encoding="utf-8") as f:
    for story in dataset:
        f.write(story['text'] + "\n<|endoftext|>\n")"""

# Only needs to be ran once

'from datasets import load_dataset\n\n# Loading only 1000 stories\ndataset = load_dataset("roneneldan/TinyStories", split="train[:1000]")\n\n# Saving everything as a text file\nwith open("data/tinystories_extract.txt", "w", encoding="utf-8") as f:\n    for story in dataset:\n        f.write(story[\'text\'] + "\n<|endoftext|>\n")'

In [2]:
"""from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

vocab_size = 2000

trainer = BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=["<|endoftext|>"],
    show_progress=True
)

file = ["data/tinystories_extract.txt"]

tokenizer.train(file, trainer=trainer)

tokenizer.save(f"weights/tiny_stories_{vocab_size}.json")
print("BPE training completed. Vocab size is :", tokenizer.get_vocab_size())"""
# Only needs to be ran once too

'from tokenizers import Tokenizer\nfrom tokenizers.models import BPE\nfrom tokenizers.trainers import BpeTrainer\nfrom tokenizers.pre_tokenizers import ByteLevel\n\ntokenizer = Tokenizer(BPE())\ntokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)\n\nvocab_size = 2000\n\ntrainer = BpeTrainer(\n    vocab_size=vocab_size,\n    special_tokens=["<|endoftext|>"],\n    show_progress=True\n)\n\nfile = ["data/tinystories_extract.txt"]\n\ntokenizer.train(file, trainer=trainer)\n\ntokenizer.save(f"weights/tiny_stories_{vocab_size}.json")\nprint("BPE training completed. Vocab size is :", tokenizer.get_vocab_size())'

In [1]:
import numpy as np
from src.ai_lib import layers
from src.ai_lib import Model, metrics, SequenceDataLoader
from src.ai_lib import optimizers
from src.ai_lib import losses
from src.ai_lib import preprocessing

vocab_size = 2000

tokenizer = preprocessing.CustomBPETokenizer(path= f"weights/tiny_stories_{vocab_size}.json")

with open("data/tinystories_extract.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

import numpy as np
data_tensor = np.array(tokenizer.encode(text_data), dtype=np.int32)

print(tokenizer.vocab_size)

2000


In [2]:
train_test_split = 0.9

n = int(train_test_split * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

batch_size = 32
block_size = 80  # Numbers of tokens to look back at
steps_per_epoch = 50 

# Creating dataloaders
train_loader = SequenceDataLoader(
    data=train_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=steps_per_epoch
)

val_loader = SequenceDataLoader(
    data=val_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=10 # Less steps for validation
)

In [3]:
from functools import partial
d_model = 128
n_heads = 4
d_ff = 512

dropout_rate = 0.1

vocab_size = tokenizer.vocab_size

norm_builder = partial(layers.RMSNorm, n_features=d_model)

attn_builder = partial(layers.RoPEMultiAttentionHead, n_heads=n_heads, d_model=d_model, is_causal=True, block_size=block_size)

ffn_builder = partial(layers.SwiGLU, in_features=d_model, hidden_features=d_ff)

llm_network = layers.Sequential([
    layers.Embedding(vocab_size=vocab_size, d_model=d_model),

    # 6 Transformers
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=0.1),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=0.1),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=0.1),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=0.1),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=0.1),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=0.1),

    layers.RMSNorm(n_features=d_model),

    layers.Linear(d_model, n_out=vocab_size)
])

model = Model(llm_network)

In [5]:
learning_rate = 1e-3
weight_decay = 1e-2

optimizer = optimizers.Adam(learning_rate=learning_rate, weight_decay=weight_decay)
loss = losses.SoftmaxCrossEntropy()

In [20]:
optimizer.lr = 5e-4
# In order to modify the learning rate in between multiple runs of the model.fit cell,
# only change it that way, do not create a new instance of the optimizer as it would delete momentum etc

In [27]:
print("Beginning of training of microgpt_v3")
model.fit(
    dataloader=train_loader,
    epochs=30,
    loss=loss,
    optimizer=optimizer,
    validation_dataloader=val_loader,
    verbose=True,
    metrics=[metrics.accuracy]
)
model.save_weights("weights/microgpt_v3.npz")

Beginning of training of microgpt_v3

--- Epoch : 0 ---
Training loss : 2.06032
  accuracy (train) : 0.51140
Validation loss : 3.14521
  accuracy (val) : 0.38394

--- Epoch : 1 ---
Training loss : 2.03207
  accuracy (train) : 0.51596
Validation loss : 3.15885
  accuracy (val) : 0.38164

--- Epoch : 2 ---
Training loss : 2.04070
  accuracy (train) : 0.51277
Validation loss : 3.22803
  accuracy (val) : 0.38375

--- Epoch : 3 ---
Training loss : 2.00980
  accuracy (train) : 0.51884
Validation loss : 3.25248
  accuracy (val) : 0.37387

--- Epoch : 4 ---
Training loss : 1.98474
  accuracy (train) : 0.52223
Validation loss : 3.26538
  accuracy (val) : 0.38570

--- Epoch : 5 ---
Training loss : 1.98263
  accuracy (train) : 0.52350
Validation loss : 3.24211
  accuracy (val) : 0.38320

--- Epoch : 6 ---
Training loss : 1.97690
  accuracy (train) : 0.52529
Validation loss : 3.24488
  accuracy (val) : 0.37715

--- Epoch : 7 ---
Training loss : 1.94231
  accuracy (train) : 0.52959
Validation loss 

In [45]:
def generate_text(model, tokenizer, prompt_text="A ", max_new_tokens=100):
    """
    Generates text
    """
    # Important : Turns off setting
    model.sequential.set_training(False)
    
    # Use of kv_cache for inference
    model.sequential.set_use_cache(True)
    model.sequential.reset_cache()
    
    # Encodes context (initial text / prompt)
    context_ids = tokenizer.encode(prompt_text)
    
    # Prefill
    X_input = np.array([context_ids], dtype=np.int32)
    logits = model.predict(X_input)
    
    # We take the predicted char and add it to the list
    next_char_id = sample_top_k(logits[0, -1, :], temperature=1.0, top_k=3)
    context_ids.append(next_char_id)
    
    # Decode
    for _ in range(max_new_tokens - 1):
        # X_input becomes a simple integer as we use kv_cache
        X_input = np.array([[next_char_id]], dtype=np.int32)
        
        logits = model.predict(X_input)
        
        # Since X_input is of length one, logits has shape (1, 1, Vocab_size)
        next_char_id = sample_top_k(logits[0, -1, :], temperature=1, top_k=3)
        context_ids.append(next_char_id)
        
    return tokenizer.decode(context_ids)


def sample_top_k(logits, temperature=1, top_k=5):
    """
    Sampling from only the k most likely tokens with temperature
    """

    logits = logits / temperature
    
    indices_to_remove = np.argsort(logits)[:-top_k]
    
    logits[indices_to_remove] = -np.inf
    
    exp_preds = np.exp(logits - np.max(logits))
    preds = exp_preds / np.sum(exp_preds)
    
    return np.random.choice(len(logits), p=preds)

generated_text = generate_text(model, tokenizer, prompt_text="Once upon a time", max_new_tokens=200)
print(generated_text)

Once upon a time, there was a little girl named Lily. Lily was very happy and played with her toys. One day, she went on a big adventure to play with her friends. One day, she saw a shiny new toy that she was on an adventure. 

Lily's mom told her that she had to stay in the mirror. She said she would go to the store and get her helicopter for a few days. The little girl was very happy and she was always sleepy to know what to do.

One day, the little girl had a wonderful idea. She decided to go on an adventure, so she went to the park to play. The little girl was so happy that she could fill things up.

The little girl wanted to show her toys to the modest car she had made. The disgusting flaging a special passport. The girl wanted to explore the world and find something new and exciting to explore.

One day


In [ ]:
model.save_weights("weights/microgpt_v3.npz")

Model saved : weights/microgpt_v3.npz
